# From image to logits, with shapes

Run locally on CPU. Start Jupyter/VS Code with the project environment.
Each operation explains which axis represents batch, sequence, heads or channels.
The notebook contains only a walkthrough; reusable implementations live in src.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import torch
from src.models.patch_embed import PatchEmbedding
from src.models.pos_encoding import PositionalEncoding
from src.models.attention import Attention
from src.models.vit import VisionTransformer
from src.utils import seed_everything
seed_everything(42)
torch.set_num_threads(2)
images = torch.randn(2, 3, 32, 32)
print('images [batch, RGB, height, width]:', images.shape)

## Patch embedding

A stride-4, kernel-4 convolution sees a non-overlapping 4×4 RGB patch at
each location. The grid is 8×8. Flattening the two grid axes gives 64 patches;
transposing moves embedding channels to the last axis, required by Linear/LayerNorm.

In [ ]:
patch = PatchEmbedding(image_size=32, patch_size=4, embed_dim=64)
grid = patch.proj(images)
print('conv grid [B,D,H/P,W/P]:', grid.shape)
flat = grid.flatten(2)
print('flatten grid [B,D,N]:', flat.shape)
tokens = flat.transpose(1, 2)
print('patch tokens [B,N,D]:', tokens.shape)
assert torch.equal(tokens, patch(images))
explicit = PatchEmbedding(embed_dim=64, method='unfold')
with torch.no_grad():
    explicit.proj.weight.copy_(patch.proj.weight.flatten(1))
    explicit.proj.bias.copy_(patch.proj.bias)
torch.testing.assert_close(tokens, explicit(images), atol=1e-6, rtol=1e-5)

## CLS and positions

CLS is one learned vector shared across images. Expanding its batch axis avoids
making separate parameters per sample; concatenation adds one sequence position.
Position addition preserves the tensor shape and distinguishes otherwise unordered patches.

In [ ]:
cls = torch.zeros(1, 1, 64)
tokens = torch.cat([cls.expand(2, -1, -1), tokens], dim=1)
print('with CLS [B,N+1,D]:', tokens.shape)
positions = PositionalEncoding(8, 64, 'learned', True)
tokens = positions(tokens)
print('with positions:', tokens.shape)

## Head split, attention and merge

Fused QKV stores three independent linear projections in one matrix. Splitting
heads divides channels, not tokens. Every head attends to all 65 tokens.
The score's last two dimensions are query and key positions. Softmax over keys
makes each query's row sum to one. Values produce a weighted feature per query.

In [ ]:
attention = Attention(64, 4)
b, n, d = tokens.shape
projected = attention.qkv(tokens)
print('QKV projected [B,N,3D]:', projected.shape)
shaped = projected.reshape(b, n, 3, 4, 16)
print('reshape [B,N,3,H,Dh]:', shaped.shape)
q, k, v = shaped.permute(2, 0, 3, 1, 4).unbind(0)
print('each Q/K/V [B,H,N,Dh]:', q.shape)
scores = (q @ k.transpose(-2, -1)) * (16 ** -0.5)
print('scores [B,H,N,N]:', scores.shape)
probabilities = scores.softmax(dim=-1)
torch.testing.assert_close(probabilities.sum(-1), torch.ones(b, 4, n))
weighted = probabilities @ v
print('weighted values [B,H,N,Dh]:', weighted.shape)
merged = weighted.transpose(1, 2).reshape(b, n, d)
print('merged [B,N,D]:', merged.shape)
output = attention.proj(merged)
torch.testing.assert_close(output, attention(tokens))

## Residual blocks and classification

Pre-LayerNorm normalizes only the embedding axis for each token. Both attention
and the GELU MLP return D channels, so residual additions preserve [B,N,D].
CLS selection reduces the sequence axis; the head maps D features to K class logits.
Logits are raw scores: use cross entropy during training, softmax only for probabilities.

In [ ]:
model = VisionTransformer(embed_dim=64, depth=2, num_heads=4)
logits, maps = model(images, return_attention=True)
print('logits [B,K]:', logits.shape)
print('layer attention [B,H,N,N]:', [value.shape for value in maps])
assert logits.shape == (2, 10)